<a href="https://colab.research.google.com/github/Peeyusj/makemore9_10week/blob/main/makemore9_10week.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import torch
import torch.nn.functional as F

# Tiny fake dataset - 5 examples, 4 possible characters
torch.manual_seed(42)

# Pretend these are outputs from your MLP's last layer
# Shape: (5 examples, 4 classes)
logits = torch.randn(5, 4)

# Step 1: Convert logits to probabilities
probs = torch.softmax(logits, dim=1)

# Step 2: Define correct answers for each example
correct = torch.tensor([0, 3, 1, 2, 0])

# Step 3: Compute loss
loss = F.cross_entropy(logits, correct)

print("logits:", logits)
print("probs:", probs)
print("loss:", loss.item())



logits: tensor([[ 1.9269,  1.4873,  0.9007, -2.1055],
        [-0.7581,  1.0783,  0.8008,  1.6806],
        [ 0.3559, -0.6866, -0.4934,  0.2415],
        [-0.2316,  0.0418, -0.2516,  0.8599],
        [-0.3097, -0.3957,  0.8034, -0.6216]])
probs: tensor([[0.4950, 0.3189, 0.1774, 0.0088],
        [0.0426, 0.2671, 0.2024, 0.4879],
        [0.3742, 0.1319, 0.1601, 0.3338],
        [0.1594, 0.2095, 0.1563, 0.4748],
        [0.1756, 0.1612, 0.5346, 0.1286]])
loss: 1.4083935022354126


In [3]:
input = torch.randn(5, 3)
one_hot = torch.zeros(5, 4)
one_hot[range(5), correct] = 1
n = probs.shape[0]
dL_dlogits = (probs - one_hot) / n
dL_dW = input.T @ dL_dlogits
dL_db = dL_dlogits.sum(dim=0)


# Rerun forward pass but tell PyTorch to track gradients
logits_pt = logits.clone().requires_grad_(True)
loss_pt = F.cross_entropy(logits_pt, correct)
loss_pt.backward()

# PyTorch's gradient for logits
print("PyTorch dL_dlogits:", logits_pt.grad)
print("Manual  dL_dlogits:", dL_dlogits)

PyTorch dL_dlogits: tensor([[-0.1010,  0.0638,  0.0355,  0.0018],
        [ 0.0085,  0.0534,  0.0405, -0.1024],
        [ 0.0748, -0.1736,  0.0320,  0.0668],
        [ 0.0319,  0.0419, -0.1687,  0.0950],
        [-0.1649,  0.0322,  0.1069,  0.0257]])
Manual  dL_dlogits: tensor([[-0.1010,  0.0638,  0.0355,  0.0018],
        [ 0.0085,  0.0534,  0.0405, -0.1024],
        [ 0.0748, -0.1736,  0.0320,  0.0668],
        [ 0.0319,  0.0419, -0.1687,  0.0950],
        [-0.1649,  0.0322,  0.1069,  0.0257]])


In [4]:
# fake hpreact - 5 examples, 3 neurons
hpreact = torch.randn(5, 3)

# forward pass
h = torch.tanh(hpreact)

# fake gradient arriving from next layer
dL_dh = torch.randn(5, 3)
dL_dhpreact = dL_dh * (1 - h**2)

# verify against PyTorch
hpreact_pt = hpreact.clone().requires_grad_(True)
h_pt = torch.tanh(hpreact_pt)
(h_pt * dL_dh).sum().backward()

print("Manual  dL_dhpreact:", dL_dhpreact)
print("PyTorch dL_dhpreact:", hpreact_pt.grad)

Manual  dL_dhpreact: tensor([[-0.3530, -0.4620, -0.1414],
        [-1.0693,  1.0870,  0.6560],
        [-1.5298,  0.0155,  0.1009],
        [ 0.5796, -0.4990, -0.0470],
        [ 1.5195,  0.0994, -0.0358]])
PyTorch dL_dhpreact: tensor([[-0.3530, -0.4620, -0.1414],
        [-1.0693,  1.0870,  0.6560],
        [-1.5298,  0.0155,  0.1009],
        [ 0.5796, -0.4990, -0.0470],
        [ 1.5195,  0.0994, -0.0358]])
